# Theory

Time series is an extrapolation problem, where the current prediction depends upon the previous prediction. So over time we get more and more unsure and the error accumulates or increases as previous day might also be a prediction :)


Decomposable: 
- Trend
- Seasonal(ity)
- Residuals

stationary - no trend or seasonal components

dickey-fuller test

Arima - Auto Regressive Ingerated Moving Average

Sarima - Seasional Arima

Varimax - Vector in Vector out Arima

(best)
Sarimax - Multivariate Seasional 

seq2seq model - LSTM

Famous Package - facebook profet

# Code

## Time Series Analysis using ARIMA

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
df = pd.read_csv(r"C:\Users\AmitLuhar\training\dataset\day25\Electric_Production.csv")
df = df.rename(columns={'DATE':'ds','IPG2211A2N':'ts'})
df.head()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(df.ts)

## Signal Decomposition (Trend, Seasonality and Residual)

Log Transform

In [ ]:
def log_transform(df,ts): return df[ts].apply(lambda x:np.log(x))

In [ ]:
# df['ts_log'] = log_transform(df,ts='ts')
df.head()

In [ ]:
def plot_decomposition(df, ts, trend,seasonal, residual):
  f, ((ax1,ax2),(ax3,ax4)) = plt.subplots(2,2, figsize=(15,8), sharex=True )

  ax1.plot(df[ts], label="Original")
  ax1.legend(loc='best')
  ax1.tick_params(axis ='x', rotation=45)

  ax2.plot(df[trend], label="Trend")
  ax2.legend(loc='best')
  ax2.tick_params(axis ='x', rotation=45)

  ax3.plot(df[seasonal], label="Seasonal")
  ax3.legend(loc='best')
  ax3.tick_params(axis ='x', rotation=45)

  ax4.plot(df[residual], label="Residuals")
  ax4.legend(loc='best')
  ax4.tick_params(axis ='x', rotation=45)

  plt.tight_layout()
  plt.show() 

In [ ]:
decomposition = seasonal_decompose(df['ts'],period=48,extrapolate_trend=3)
new_df = df.copy()
new_df.loc[:,'trend'] = decomposition.trend
new_df.loc[:,'seasonal'] = decomposition.seasonal
new_df.loc[:,'residual'] = decomposition.resid
plot_decomposition(new_df,ts='ts',trend='trend',seasonal='seasonal',residual='residual')

### Stationarity Test: Dickey Fuller Test

It is stationarity if mean is const, std is const and there is no seasonality.

If Either of them are violated, we can **not** user AR or MA models **PERIOD**.

A BIG BUT, if we can identify unit root's and make some transformation and make it stationarity we can yes we can use ARIMA

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
dickeyFtest = adfuller(new_df['residual'],autolag='AIC')

In [ ]:
def test_stationarity(df, ts):

  rolmean = df[ts].rolling(window=12, center= False).mean()
  rolstd =  df[ts].rolling(window=12, center = False).std()
  plt.Figure(figsize=(6,4))
  orig = plt.plot(df[ts], color = 'blue', label ="Original")
  mean = plt.plot(rolmean, color ='red', label ="Rolling Mean")
  std = plt.plot(rolstd, color='black', label ="Rolling Std")
  plt.legend(loc = 'best')
  plt.title("Rolling Mean and Standard Deviation for  %s" %(ts))
  plt.xticks(rotation =45)
  plt.show(block = False)
  plt.close

  print('Results:')
  dftest = adfuller(df[ts], autolag='AIC')
  dfoutput = pd.Series(dftest[0:4], index=["Test Statistic",'p=value','# Lasgs Used',' Number of Observations'])

  for key, value in dftest[4].items():
    dfoutput['Critical Value(%s)' %key]= value

  print(dfoutput)

In [ ]:
test_stationarity(df=new_df,ts='residual')
# test_stationarity(df=new_df,ts='log_residual')

**Autocorrelation Function (AFC):** Help us determine correlation of current month with previous k month directly and in-directly 

**Partial Autocorrelation Function(PACF):** Only determine direct correlation of current entity with previous kth one

**Lags:** -1,-2,-3,k are called lags, so current entity with previous one is called 1 lag

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Create figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12,8))

# Plot the ACF of df
plot_acf(new_df['residual'], lags=100, zero=False, ax=ax1);

# Plot the PACF of df
plot_pacf(new_df['residual'], lags=100, zero=False, ax=ax2);

## Forecasting using ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
def run_Arima(df,ts,p,d,q):
  model =ARIMA(df[ts], order=(p,d,q))
  results_arima=model.fit()

  len_results =len(results_arima.fittedvalues)
  ts_modified =df[ts][-len_results:]

  rss = sum((results_arima.fittedvalues-ts_modified)**2)
  rmse = np.sqrt(rss/len(df[ts]))
  print("RMSE: ",rmse)

  plt.figure(figsize=(12,6))
  plt.plot(df[ts])
  plt.plot(results_arima.fittedvalues, color='red')
  plt.show()

  return results_arima

In [ ]:
new_df = new_df.fillna(value=0)

In [ ]:
Model_AR = run_Arima(df=new_df,ts='residual',p=15,d=0,q=0)

In [ ]:
my_forecast = Model_AR.forecast(48)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(new_df['residual'])
plt.plot(my_forecast, color = 'green')
plt.show()

In [ ]:
Model_AR.summary()

In [ ]:
Model_AR.conf_int(alpha=0.05)

In [ ]:
from statsmodels.graphics.tsaplots import plot_predict

In [ ]:
fig, ax = plt.subplots()
ax = new_df['residual'].plot(ax=ax)
plot_predict(Model_AR,390,450, ax=ax)
plt.show()